In [1]:
from platform import python_version
print(python_version())

3.11.14


### BayesPrism

#### **Bayesian cell Proportion Reconstruction** Inferred using Statistical Marginalization (BayesPrism):

A Fully Bayesian Inference of Tumor Microenvironment composition and gene expression

BayesPrism consists of 
- the deconvolution modules and 
- the embedding learning module. 

The **deconvolution module** models a prior from cell type-specific expression profiles from scRNA-seq to jointly estimate the posterior distribution of cell type composition and cell type-specific gene expression from bulk RNA-seq expression of tumor (or non-tumor) samples. 

The **embedding learning** module uses Expectation-maximization (EM) to approximate the tumor expression using a linear combination of malignant gene programs while conditional on the inferred expression and fraction of non-malignant cells estimated by the deconvolution module.


#### Ref

Human Pancreatic Cancer Single-Cell Atlas Reveals Association of CXCL10+ Fibroblasts and Basal Subtype Tumor Cells

Ian M Loveless, Nina G Steele, et al.

https://pubmed.ncbi.nlm.nih.gov/39636224/

#### Abstrac

Purpose: Pancreatic ductal adenocarcinoma (PDAC) patients with tumors enriched for the basal-like molecular subtype exhibit enhanced resistance to standard-of-care treatments and have significantly worse overall survival compared with patients with classic subtype-enriched tumors. It is important to develop genomic resources, enabling identification of novel putative targets in a statistically rigorous manner.

Experimental design: We compiled a single-cell RNA sequencing (scRNA-seq) atlas of the human pancreas with 229 patient samples aggregated from publicly available raw data. We mapped cell type-specific scRNA-seq gene signatures in bulk RNA-seq (n = 744) and spatial transcriptomics (ST; n = 22) and performed validation using multiplex immunostaining.

### Github

https://github.com/PDAC-MULTIOMIC/PDAC_Atlas

- week


### Data on Zenodo



```Bash
curl -L -C - -o scAtlas.rds.gz \
  "https://zenodo.org/records/14199536/files/scAtlas.rds.gz?download=1"

echo "705078352e1feb260a64cec67c64ade0  scAtlas.rds.gz" | md5sum -c -
gunzip -k scAtlas.rds.gz          # -> scAtlas.rds, expect ~60-100 GB uncompressed
```

In [5]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config
from libs.prism_lib import PRISM
from libs.prism_program_lib import *
from libs.prism_diagnostics_helpers import *

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/PAAD/config/all_lfc_cutoffs_PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
dstudy = 'Oh2023' # https://www.nature.com/articles/s41467-023-40895-6#Sec26
dstudy = 'Peng2018' # first study proposed by Claude
dstudy = 'scAtlas2025' # https://pubmed.ncbi.nlm.nih.gov/39636224/ - remove Peng and Metastasis


mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID, dstudy=dstudy,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/PAAD
>>> PAAD Tumor
>>> case Tumor
>>> psi_id or disease: PAAD
Error: No data available for the specified PAAD.
Error: could not find /home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc/PAAD_final_LFC_Tumor_x_CTRL_not_normalized.tsv
No dflfc table was calculated for this case Tumor

Echo Parameters:
	0/0 DEGs/ensembl.
		Up 0/0 DEGs/ensembl.
		Dw 0/0 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.


In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [ ]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

### Open primary cites from cbio

In [ ]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

### Prism Class instantitaion

https://github.com/Danko-Lab/BayesPrism

In [ ]:
prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, dstudy=dstudy, verbose=verbose)

prism.root_prism, prism.root_prism.exists()

### Getting data

obj <- readRDS("scAtlas.rds")

Look in that output for two things: 

- the study/dataset column (values like
  - Peng, 
  - Werba, 
  - Steele, 
  - or accessions PRJCA001063) 
- the disease-state column (values like 
  - PT, 
  - Met, 
  - AdjN, 
  - Donor). 
  
They won't necessarily be named study/disease_state. Once you know them, substitute below ...


#### Bash code

```Bash

root_prism='/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/scAtlas2025/prism'
cd $root_prism


# 33.4 GB
curl -L -C - -o scAtlas.rds.gz "https://zenodo.org/records/14199536/files/scAtlas.rds.gz?download=1"

echo "705078352e1feb260a64cec67c64ade0  scAtlas.rds.gz" | md5sum -c -

gunzip -k scAtlas.rds.gz          # -> scAtlas.rds, expect ~60-100 GB uncompressed


conda env list
conda activate renv
R
# R version 4.5.3 (2026-03-11) 
```

#### R code

```R
#--------- install Seurat --------------
install.packages("remotes")
library(remotes)

remotes::install_version("Seurat", version = "4.4.0")


```

### Reading scAtlas 2025

In [ ]:
import scanpy as sc, scipy.io as sio

In [7]:
root_prism=Path('/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/scAtlas2025/prism')
os.listdir(root_prism)

['atlas_counts_noPeng_noMet.mtx',
 'scAtlas.rds',
 'atlas_counts.mtx',
 'scAtlas.rds.gz',
 'genes_noPeng_noMet.txt',
 'cells.txt',
 'atlas_meta.csv',
 'cells_noPeng_noMet.txt',
 'atlas_meta_noPeng_noMet.csv',
 'genes.txt']

In [8]:
fname = "atlas_counts_noPeng_noMet.mtx"
filename = root_prism / fname

# transpose -> cells x genes
X  = sio.mmread(filename).T.tocsr()
X.shape

(521758, 36601)

In [10]:
fname = "atlas_meta_noPeng_noMet.csv"
filename = root_prism / fname

df_meta = pd.read_csv(filename, index_col=0)
print(df_meta.shape)
df_meta.head(3).T


(521758, 13)


,AAACATACTCGTTT-1_1,AAACCGTGGGTAGG-1_1,AAAGCAGACTGAGT-1_1
nCount_RNA,2480,690,1670
nFeature_RNA,962,338,623
percent.mt,12.661,2.174,2.156
seurat_clusters,4,3,3
Count,241,241,241
Study..Citation..PMID.,"Lin W, Noel P, Borazanci EH, Lee J et al. Single-cell transcriptome analysis...","Lin W, Noel P, Borazanci EH, Lee J et al. Single-cell transcriptome analysis...","Lin W, Noel P, Borazanci EH, Lee J et al. Single-cell transcriptome analysis..."
GSE.SRA..Study.,GSE154778,GSE154778,GSE154778
Name,T1,T1,T1
If.metastatic..location,NaN,NaN,NaN
Clusters,DUCTAL,FIBROBLASTS,FIBROBLASTS


In [12]:
fname = "genes_noPeng_noMet.txt"
filename = root_prism / fname
genes = [l.strip() for l in open(filename)]

fname = "cells_noPeng_noMet.txt"
filename = root_prism / fname
cells = [l.strip() for l in open(filename)]

len(genes), genes[:5], len(cells), cells[:5]


(36601,
 ['MIR1302-2HG', 'FAM138A', 'OR4F5', 'AL627309.1', 'AL627309.3'],
 521758,
 ['AAACATACTCGTTT-1_1',
  'AAACCGTGGGTAGG-1_1',
  'AAAGCAGACTGAGT-1_1',
  'AAAGGCCTGCTCCT-1_1',
  'AAATACTGTGGATC-1_1'])

### AnnData

In [14]:
ad = sc.AnnData(X, obs=df_meta.loc[cells], var=pd.DataFrame(index=genes))
ad

AnnData object with n_obs × n_vars = 521758 × 36601
    obs: 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'seurat_clusters', 'Count', 'Study..Citation..PMID.', 'GSE.SRA..Study.', 'Name', 'If.metastatic..location', 'Clusters', 'Treatment', 'DiseaseState', 'TreatmentType'

In [15]:
ad.layers["counts"] = ad.X.copy()

In [ ]:
fname_h5ad = "scAtlas_noPeng_noMet.h5ad"
filename_h5ad = root_prism / fname_h5ad
ad.write_h5ad(filename_h5ad)
